# M1 — Knowledge Gap Classifier (DeBERTa-v3-base)
Target: **F1 (micro) >= 0.85**. See `docs/phases/MENTORA_Phase4_Model_Training.md` §4.

**Two real adaptations from the plan doc, disclosed here rather than silently
changed:**
1. **Single-label, not multi-label.** The doc assumes a pipe-separated
   multi-topic `topic` column. The real data has exactly one `topic` per
   question, so this trains a standard single-label multi-class classifier
   (softmax, not sigmoid) instead of `problem_type="multi_label_classification"`.
2. **Topic labels for the 1777 MDCAT-derived questions are heuristic**
   (keyword-matched, not manually tagged — see
   `datasets/m1_gap_classifier/tag_topics.py` and `mentora_data/SOURCES.md`).
   67% of those rows fell back to a generic "General <Subject>" bucket because
   no keyword matched. The 378 Math questions have real topic/subtopic labels
   from their source dataset. **Treat the reported F1 as a first-pass signal,
   not a final number** — manually reviewing/improving the MDCAT topic tags
   would likely raise it.

## Setup — mount Drive, confirm GPU, install deps
Same first 3 cells in every Mentora training notebook. See `docs/phases/MENTORA_Phase4_Model_Training.md` §0.

In [8]:
from google.colab import drive
drive.mount('/content/drive')

import subprocess
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
# Expect to see "Tesla T4" — if this errors or shows no GPU, go to
# Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU -> Save, then re-run.

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Thu Oct  8 16:28:03 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   67C    P0             28W /   70W |    3957MiB /  15360MiB |      0%      Default |
|          

In [9]:
import os
DATA = '/content/drive/MyDrive/mentora_data'
MODELS = '/content/drive/MyDrive/mentora_models'
for m in ['model1_gap_classifier', 'model2_question_generator',
          'model3_trajectory_predictor', 'model4_career_matcher', 'model5_cv_ner']:
    os.makedirs(f'{MODELS}/{m}', exist_ok=True)

In [10]:
!pip install -q torch --index-url https://download.pytorch.org/whl/cu121
!pip install -q transformers accelerate

## Load and tokenize

In [11]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from datasets import Dataset
import pandas as pd, numpy as np

df = pd.read_csv(f'{DATA}/processed/m1/question_bank_final.csv')
df = df.dropna(subset=['question_text', 'topic'])
topics = sorted(df['topic'].unique())
NUM_TOPICS = len(topics)
topic_to_idx = {t: i for i, t in enumerate(topics)}
print(f"{NUM_TOPICS} distinct topics, {len(df)} questions")
print(df['topic'].value_counts())

df['label'] = df['topic'].map(topic_to_idx)

tokenizer = AutoTokenizer.from_pretrained('microsoft/deberta-v3-base')
def tokenize(batch):
    return tokenizer(batch['question_text'], truncation=True, padding='max_length', max_length=128)

ds = Dataset.from_pandas(df[['question_text', 'label']])
ds = ds.train_test_split(test_size=0.15, seed=42)
ds = ds.map(tokenize, batched=True)

20 distinct topics, 2155 questions
topic
General Chemistry                    381
General Biology                      357
General Physics                      233
General Logical Reasoning            216
General English                      129
Algebra                              124
Advanced Math                        107
Problem-Solving and Data Analysis     97
Mechanics                             73
Electricity & Magnetism               55
Genetics                              51
Geometry and Trigonometry             50
Physical Chemistry                    49
Waves & Optics                        48
Physiology                            44
Biochemistry                          32
Organic Chemistry                     31
Modern Physics                        27
Inorganic Chemistry                   26
Botany                                25
Name: count, dtype: int64


Map:   0%|          | 0/1831 [00:00<?, ? examples/s]

Map:   0%|          | 0/324 [00:00<?, ? examples/s]

## Model + metrics

In [12]:
import torch
model = AutoModelForSequenceClassification.from_pretrained(
    'microsoft/deberta-v3-base', num_labels=NUM_TOPICS, torch_dtype=torch.float32
)

from sklearn.metrics import f1_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "f1_micro": f1_score(labels, preds, average="micro", zero_division=0),
        "f1_macro": f1_score(labels, preds, average="macro", zero_division=0),
    }

# Rare topics were merged into their subject's General bucket (attempt 1
# had 30 classes, several under 10 examples; now 20, all >=25 -- see
# datasets/m1_gap_classifier/merge_rare_topics.py), but the surviving
# classes still range from 25 to 381 examples. Inverse-frequency class
# weights keep the trainer from just learning to always predict the
# biggest classes.
class_counts = df['label'].value_counts().sort_index()
class_weights = torch.tensor(
    (len(df) / (NUM_TOPICS * class_counts)).values, dtype=torch.float32
).to(model.device)

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop('labels')
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fct = torch.nn.CrossEntropyLoss(weight=class_weights.to(logits.device))
        loss = loss_fct(logits.view(-1, NUM_TOPICS), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
pooler.dense.weight                     | MISSING    | 
pooler.dense.bias                       | MISSING    | 
classifier.bias                         | MISSING    | 
classifier

## Train — epoch checkpointing + auto-resume

> **Before re-running from here:** attempt 1 trained on 30 topic classes (several with under 10 examples) and got F1 (micro) = 0.4352 vs the >=0.85 target. Rare classes are now merged into their subject general bucket (30 -> 20 classes, all >=25 examples -- see `datasets/m1_gap_classifier/merge_rare_topics.py`), and the cell above now trains with inverse-frequency class weighting. The training cell below resumes from `mentora_models/model1_gap_classifier/checkpoint-*` on Drive if it exists -- that checkpoint has a 30-class output head and won't load into the new 20-class model. **Delete that checkpoint folder on Drive before re-running**, or the resume step will error on a shape mismatch. Realistic expectation: this gets F1 meaningfully higher but likely still short of 0.85 -- closing the rest of the gap needs the MDCAT topic labels hand-corrected, not just rebalanced.

In [13]:
OUT_DIR = f'{MODELS}/model1_gap_classifier'

args = TrainingArguments(
    output_dir=OUT_DIR,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=5,
    learning_rate=2e-5,
    save_strategy="epoch",
    eval_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_micro",
    fp16=True,
)

trainer = WeightedTrainer(
    model=model, args=args,
    train_dataset=ds['train'], eval_dataset=ds['test'],
    compute_metrics=compute_metrics,
)

import glob
existing_checkpoints = glob.glob(f'{OUT_DIR}/checkpoint-*')
resume = sorted(existing_checkpoints)[-1] if existing_checkpoints else None
trainer.train(resume_from_checkpoint=resume)

Epoch,Training Loss,Validation Loss,F1 Micro,F1 Macro
1,No log,2.513100,0.222222,0.093653
2,No log,2.051180,0.268519,0.229430
3,2.462673,1.673892,0.419753,0.379956
4,2.462673,1.499519,0.422840,0.403619
5,1.499526,1.453194,0.469136,0.441176


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1145, training_loss=1.8777106405866197, metrics={'train_runtime': 321.9906, 'train_samples_per_second': 28.433, 'train_steps_per_second': 3.556, 'total_flos': 602303551288320.0, 'train_loss': 1.8777106405866197, 'epoch': 5.0})

## Save final model

In [14]:
trainer.save_model(f'{OUT_DIR}/final')
tokenizer.save_pretrained(f'{OUT_DIR}/final')
import json
with open(f'{OUT_DIR}/final/topic_labels.json', 'w') as f:
    json.dump(topics, f)  # save the label->topic mapping alongside the model — needed for inference
print(trainer.evaluate())

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training Loss,Validation Loss,Epoch,F1 Micro,F1 Macro
1.499526,1.453194,5,0.469136,0.441176


{'eval_loss': 1.4531939029693604, 'eval_f1_micro': 0.4691358024691358, 'eval_f1_macro': 0.44117591840407105}


## If you miss the target (F1 micro < 0.85)
- This is very likely the heuristic-topic-label noise described above, not a
  training problem — check per-class F1 and see if the "General <Subject>"
  fallback classes (which mix unrelated questions together) are dragging the
  average down specifically
- If time allows, hand-correcting even just the MDCAT rows' topics (or
  merging the noisiest "General X" buckets back into subject-level) is the
  highest-leverage fix, more so than hyperparameter tuning
- Otherwise: sweep the classification threshold isn't applicable here (single-label
  softmax, not multi-label) — try `learning_rate=1e-5` or `num_train_epochs=8`
  if it looks like underfitting rather than label noise